In [ ]:
from randomness import *
from simulation_exact import *
from utils import *
from data_analysis import *
from main import *
import numpy as np
import warnings
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")
seed = 42
np.random.seed(seed)
rng = np.random.default_rng(seed)

In [ ]:
def generate_data_d(n, mu1, mu2, rng, sigma = 1.0, d=1):
    n_per_class = n // 2
    X1 = rng.normal(loc=mu1, scale=sigma, size=(n_per_class, d))
    X2 = rng.normal(loc=mu2, scale=sigma, size=(n_per_class, d))
    X = np.vstack([X1, X2])
    y = np.concatenate([
        -1 * np.ones(n_per_class),
        np.ones(n_per_class)
    ])

    v = np.ones(n)

    return X, y, v

In the block below, start by setting the parameters for the learning model  (lines 1-10 in block below)

experiments: (two blocks down )

1. to run the expirement for the same sample while changing mu_1 - mu_2, <br>
    remove the comments in lines 12 and 17 <br>
    comment line 16 <br>
    set T = 1 <br>

2. to run the expirement for the different samples and averaging over each T samples , <br>
    comments  lines 12 and 17 <br>
    remove comment line 16 <br>
    set T to be number of samples to be drawn per mu (line 12) <br>
 
In both cases you can choose parameters in lines 11-13, in the block below: <br>
    ds:= different dimensions to run the expierments on (line 11) <br>
    mus:= different mus to run the experiments on, where one distrbution has mean mu and the other has -mu (line 13)
    

In [ ]:
# 1. choose parameters 
n = 100 # number of data points in a sample
use_loss = 'hinge' #'log' or 'hinge' or 'squared_hinge' 
sigma_loss = 1.0 #TODO: integrate in code 
c = 1.0 # choose c without scaling 
show_plots = False # set to True to see plots of the binary search
is_throw = True # set to True to throw out points outside margin --> only when loss is lipschitz. 
# TODO: k 
# TODO: fit_intercept 
# ---------------------------------
ds = [2, 4, 6, 10, 12] # dimensions to run experiments on
T = 500 # number of trials
mus = np.linspace(1.0, 0.0, 31) # different mus to run the experiments on
sigma = 1.0 # data sd for generating process 


In [ ]:
# 3. run exact simulation
# returns df with cols "agent" | "true_v" | "critical_v" | "allocation" | "welfare" | "utility"
# utility = welfare - payment
# welfare = allocation * true_v
# critical_v = payment 

def compute_num_payers_for_d(d):
    num_payers = np.zeros(len(mus))
    # x, y, v = generate_data_d( n, mu, -mu, rng, sigma, d)
        
    for t in range(T):
        for i, mu in enumerate(mus):
            x, y, v = generate_data_d( n, mu, -mu, rng, sigma, d) 
            #x_updated = update_data(x, y, mu1, mu2)
            df_exact, _ = run_exact(x, y, v, c, use_loss, plot=False)
            if df_exact is not None:
                num_payers[i] += len(df_exact)

    return d, num_payers / T


In [ ]:
results = Parallel(n_jobs=-1, backend="loky")(
    delayed(compute_num_payers_for_d)(d) for d in ds
)

results = dict(results)  # {d: num_payers}
for d, num_payers in results.items():
    plt.plot(mus, num_payers, label=f"d={d}")

plt.xlabel("μ")
plt.ylabel("Average number of payers")
plt.legend()
plt.grid(True)
plt.show()